<a href="https://colab.research.google.com/github/kyubiepbl/auditoria-copasa-mg-2025/blob/main/gerador_base_copasa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Configurações Iniciais
NUM_LINHAS = 500000
SALDO_ALVO = 7179980000.00 # Saldo Consolidado do Intangível 2025
MATRICULA = 222020687
np.random.seed(MATRICULA)
random.seed(MATRICULA)

print("Gerando base simulada para a COPASA - Ativo Intangível...")

# Geração de dados base
tipos_custo = ['Material de Construção', 'Mão de Obra Própria', 'Serviços de Terceiros (Medição)', 'Licenciamento Ambiental', 'Equipamentos']
municipios_mg = ['Belo Horizonte', 'Contagem', 'Betim', 'Uberlândia', 'Juiz de Fora', 'Montes Claros', 'Divinópolis', 'Ipatinga']

data_inicio = datetime(2025, 1, 1)
datas = [data_inicio + timedelta(days=random.randint(0, 364)) for _ in range(NUM_LINHAS)]

df = pd.DataFrame({
    'ID_Registro': range(1, NUM_LINHAS + 1),
    'Data_Lancamento': datas,
    'Contrato_Concessao': np.random.choice([f'CONC-{random.randint(100, 999)}/20{random.randint(10,25)}' for _ in range(50)], NUM_LINHAS),
    'Municipio': np.random.choice(municipios_mg, NUM_LINHAS),
    'Tipo_Custo': np.random.choice(tipos_custo, NUM_LINHAS, p=[0.4, 0.2, 0.3, 0.05, 0.05]),
    'Nota_Fiscal_Referencia': [f"{random.randint(1000, 999999)}" if random.random() > 0.2 else "ISENTO" for _ in range(NUM_LINHAS)],
    'Valor_BRL': np.random.lognormal(mean=7.5, sigma=1.5, size=NUM_LINHAS) # Distribuição assimétrica típica de custos
})

# Ajustar o valor para bater EXATAMENTE com o balanço
fator_correcao = SALDO_ALVO / df['Valor_BRL'].sum()
df['Valor_BRL'] = (df['Valor_BRL'] * fator_correcao).round(2)

# Garantir a exatidão dos centavos na última linha
diferenca = SALDO_ALVO - df['Valor_BRL'].sum()
df.loc[df.index[-1], 'Valor_BRL'] += diferenca
df['Valor_BRL'] = df['Valor_BRL'].round(2)

# ==========================================
# INSERÇÃO DE ERROS (COM REGISTRO PARA PT 530)
# ==========================================
erros_log = []

# Erro 1: Lançamento duplicado (Fraude/Erro de sistema)
idx_dup = random.randint(0, NUM_LINHAS - 10)
df.loc[idx_dup + 1] = df.loc[idx_dup]
df.loc[idx_dup + 1, 'ID_Registro'] = idx_dup + 2
erros_log.append(f"Duplicidade exata de valores e metadados nos IDs {idx_dup+1} e {idx_dup+2}.")

# Erro 2: Valores negativos no ativo (Inconsistência contábil)
idx_neg = random.sample(range(NUM_LINHAS), 5)
for idx in idx_neg:
    df.loc[idx, 'Valor_BRL'] = -abs(df.loc[idx, 'Valor_BRL'])
    erros_log.append(f"Valor negativo inserido no ID {df.loc[idx, 'ID_Registro']}.")

# Erro 3: Lançamento em final de semana/feriado com alto valor (Risco de Fraude - PT 240)
idx_fds = random.choice(range(NUM_LINHAS))
df.loc[idx_fds, 'Data_Lancamento'] = datetime(2025, 12, 25) # Natal
df.loc[idx_fds, 'Valor_BRL'] = 5000000.00
erros_log.append(f"Lançamento atípico no feriado (25/12/2025) com valor material no ID {df.loc[idx_fds, 'ID_Registro']}.")

# Re-ajustar total após erros para manter conciliação
fator_correcao_final = SALDO_ALVO / df['Valor_BRL'].sum()
df['Valor_BRL'] = (df['Valor_BRL'] * fator_correcao_final).round(2)
diferenca_final = SALDO_ALVO - df['Valor_BRL'].sum()
df.loc[df.index[-1], 'Valor_BRL'] += diferenca_final

# Salvar Arquivos
df.to_csv('copasa_intangivel_2025_simulado.csv', index=False)
with open('log_erros_inseridos.txt', 'w') as f:
    f.write("RELATÓRIO DE ERROS INSERIDOS PROPOSITADAMENTE\n\n")
    for erro in erros_log:
        f.write(f"- {erro}\n")

print("Sucesso! Arquivo 'copasa_intangivel_2025_simulado.csv' gerado.")